In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("catalog", "")
catalog = dbutils.widgets.get("catalog")

In [0]:
city_df= (spark.read.format("csv")
    .option("header", "true")
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .load("/Volumes/transportation/source_data/city_data")
)


In [0]:

city_df.withColumn("file_path", col("_metadata.file_path"))


In [0]:
city_df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.bronze_transport.city")

In [0]:
city_df.display()

In [0]:
city_df.printSchema()

In [0]:

trips_schema = StructType([
    StructField("trip_id", StringType(), True),
    StructField("trip_date",StringType(), True),
    StructField("city_id", StringType(), True),
    StructField("passenger_type", StringType(), True),
    StructField("distance_travelled", StringType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("passenger_rating", DoubleType(), True),
    StructField("driver_rating", DoubleType(), True)
])

trips_df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("header", True) \
    .schema(trips_schema) \
    .load("/Volumes/transportation/source_data/trips_data")



In [0]:
trips_df = trips_df.withColumn("file_path", col("_metadata.file_path")) \
                   .withColumn("ingest_timestamp", current_timestamp())

In [0]:
trips_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .option("checkpointLocation", f"/Volumes/{catalog}/bronze_transport/bronze_check_point_location") \
    .table(f"{catalog}.bronze_transport.trips_bronze")